In [1]:
# Library imports
import torch
import math
import random
import json
import tqdm
import os
import spacy
import re

import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

In [2]:
# Load env settings

env = dotenv_values(DOTENV_PATH)

In [3]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 4070 Ti


In [4]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [5]:
# Configuration

DETECT_MODE = 'claims'
CLAIMS_INPUT_DIR = './extracted_claims/pass_02_temporal_fixes'

NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

# Standard generation parameters used by all runs
CLAIM_MODEL_NAME = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
CLAIM_MODEL_TEMPERATURE = 0.1
CLAIM_MODEL_MAX_NEW_TOKENS = 4096
CLAIM_MODEL_BACKEND = 'local'
CLAIM_MODEL_VERBOSE = True

In [6]:
# Create models

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

nlp = spacy.load("en_core_web_sm")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Load dataset

dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

# Label examples
for example in dataset['pos'].values():
	example['label'] = 1.0 

for example in dataset['neg'].values():
	example['label'] = 0.0 

# Flatten list
all_examples = { **dataset['pos'], **dataset['neg'] }

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [16]:
# Model functions

def normalize_punctuation(text: str) -> str:
    # remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # ensure space after punctuation if followed by word
    text = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", text)

    # normalize quotes spacing
    text = re.sub(r"\s+([\"'])", r" \1", text)
    text = re.sub(r"([\"'])\s+", r"\1 ", text)

    # normalize dash spacing
    text = text.replace(" - ", "-")

    # collapse whitespace
    #text = re.sub(r"\s+", " ", text)

    return text.strip()

def remove_structural_lines(text):
    lines = text.split("\n")
    clean = []
    
    for line in lines:
        l = line.strip()

        if not l:
            continue

        # wikipedia headers
        if re.match(r"^\s*(=+\s*)+[^=]+?(\s*=)+\s*$", l):
            continue

        # markdown headers
        if re.match(r"^#+\s", l):
            continue

        # numbered lists
        if re.match(r"^\d+\s*[.)]\s*", l):
            continue

        # bullet lists
        if re.match(r"^[-*]\s+", l):
            continue

        # table rows
        if "|" in l and l.count("|") >= 2:
            continue

        clean.append(l)

    return " ".join(clean)

def normalize_numbers(text: str) -> str:
    # Weird commas
    text = text.replace("@,@", ",")

    # decimal numbers: "2 . 9" -> "2.9"
    text = re.sub(r"(\d)\s*\.\s*(\d)", r"\1.\2", text)

    # thousands separators: "1 , 000" -> "1,000"
    text = re.sub(r"(\d)\s*,\s*(\d)", r"\1,\2", text)

    # currency spacing "$ 2.9" -> "$2.9"
    text = re.sub(r"([$€£])\s+(\d)", r"\1\2", text)

    return text

def normalize_brackets(text: str) -> str:
    # "( text )" -> "(text)"
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)

    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    text = re.sub(r"\{\s+", "{", text)
    text = re.sub(r"\s+\}", "}", text)

    return text

def has_verb(text):
    doc = nlp(text)
    return any(token.pos_ == "VERB" for token in doc)

def split_sentences(text):
    text = normalize_punctuation(text)
    text = remove_structural_lines(text)
    text = normalize_numbers(text)
    text = normalize_brackets(text)

    doc = nlp(text)
    sentences = [s.text.strip() for s in doc.sents]

    def sentence_filter(sentence):
        # min word count
        if len(sentence.split()) < 4:
            return False
        
        # contain actual words
        if re.match(r"^[\d\s.,-]+$", sentence):
            return False
        
        # requires verb
        if not has_verb(sentence):
            return False
        
        return True

    return list(filter(sentence_filter, sentences))

def argtopk(a, k):
    idx = np.argpartition(a, -k)[-k:]
    return idx[np.argsort(a[idx])[::-1]]

def get_sentence_clusters(sentences, embeddings, top_k):
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0  # Exclude self-similarity by setting diagonal to -1

    cluster_indices = [ ]
    for i in range(len(sentences)):
        cluster_indices.append(argtopk(similarity_matrix[i], top_k))

    clusters = [ ]
    for i, neighbours in enumerate(cluster_indices):
        cluster = [ sentences[i], *[ sentences[j] for j in neighbours ]]
        clusters.append(cluster)

    return clusters

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    
def detect(id, document, mode, verbose):
    if mode == 'claims':
        file_name = os.path.join(CLAIMS_INPUT_DIR, f'{id}.txt')
        if os.path.isfile(file_name):
            with open(file_name, 'r', encoding='utf-8') as file:
                claims = file.readlines()
        else:
            # claims = extract_claims(
            #     text=document,
            #     model_name=CLAIM_MODEL_NAME,
            #     backend=CLAIM_MODEL_BACKEND,
            #     temperature=CLAIM_MODEL_TEMPERATURE,
            #     max_new_tokens=CLAIM_MODEL_MAX_NEW_TOKENS,
            #     verbose=CLAIM_MODEL_VERBOSE,
            # )

            # Skip non-cached examples for now
            return None, None, None

    elif mode == 'sentences':
        claims = split_sentences(document)
    else:
        raise "Invalid mode"

    if verbose:
        print()
        print(f'Extracted {len(claims)} claims:')
        print('\n'.join(claims))

    if len(claims) <= 1:
        return None, None, None
    
    embeddings = embed_model.encode(claims)

    single_claims = [ [ claim ] for claim in claims ]
    clustered_claims = get_sentence_clusters(claims, embeddings, 2)
    clusters = [ *single_claims, *clustered_claims ]

    p_contra_max = 0.0
    evidence = []
    evidence_max = None

    for cluster in clusters:
        if len(cluster) > 1:
            p_contra = 0

            for i in range(len(cluster)):
                premise_claims = [ claim for j, claim in enumerate(cluster) if i != j ]
                premise = ' '.join(premise_claims)
                hypothesis = cluster[i]
            
                prediction = predict(premise, hypothesis)
                p_contra += prediction['contradiction']

                if verbose:
                    print(f'Comparing:')
                    
                    print(f'  {premise}')
                    print(f'  {hypothesis}')

                    print()

                    print(f'  Prediction: {prediction["contradiction"]:.4f}')
                    print()
            
            p_contra = p_contra / len(cluster)
        else:
            prediction = predict(cluster[0], "")
            p_contra = prediction['contradiction']

            if verbose:
                print(f'Testing:')
                print(f'  {cluster[0]}')
                print(f'  Prediction: {p_contra:.4f}')
                print()

        if p_contra > p_contra_max:
            evidence_max = (cluster, p_contra)
            p_contra_max = p_contra

        if p_contra > 0.70:
            evidence.append((cluster, p_contra))

    return p_contra_max, evidence_max, evidence

In [9]:
# NLI test

contra = "Holmes grows angry when Watson touches items explaining that he doesn't mind his things being touched"

premise = contra
hypothesis = ""

print(predict(premise, hypothesis))

{'entailment': 0.0011224746704101562, 'neutral': 0.0039215087890625, 'contradiction': 0.9951171875}


In [15]:
# Single evaluation test

pos = True					# Test example from positive or negative set
example_id = None	# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585
# wiki_train_29443

if example_id is None:
	example_ids = list((dataset['pos' if pos else 'neg']).keys())
	example_id = example_ids[random.randint(0, len(example_ids) - 1)]

example = all_examples[example_id]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if 'evidence' in example:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence_max, all_evidence = detect(example['unique id'], example['text'], DETECT_MODE, True)
p_contra, evidence_max, all_evidence

Detecting contradictions in example 3489738270_2

Abu Dhabi, United Arab Emirates Lost luggage after a long flight is a common, frustrating occurrence of modern air travel. And sometimes, airlines lose things that are irreplaceable. American Jennifer Stewart says she was devastated to learn that Etihad Airways lost her most important baggage following a recent trip from Abu Dhabi to New York City: her 2-year-old pet cat, Felix. Stewart said that she and her husband, Joseph Naaman, booked Felix on their Etihad Airways flight from the United Arab Emirates to New York's John F. Kennedy International Airport on April 1.Shortly after the plane arrived in New York that evening, Felix was found somewhere on the grounds of Kennedy Airport, according to Etihad Airways.Felix was able to get loose, Stewart said she believes, because the cat's plastic carrier was badly damaged at some point either during the flight or the transfer from the airplane to the pickup area. Stewart said a cargo manager 

(0.9978841145833334,
 (['At some point in time, Jennifer Stewart said that people assume they pay extra to have their pets taken care of.\n',
   'At some point in time, Jennifer Stewart said that pets are treated no differently than a free piece of checked luggage.\n',
   'At some point in time, Jennifer Stewart said that people trust that others care and are doing their job well.\n'],
  0.9978841145833334),
 [(['At some point in time, Jennifer Stewart said that people assume they pay extra to have their pets taken care of.\n',
    'At some point in time, Jennifer Stewart said that pets are treated no differently than a free piece of checked luggage.\n',
    'At some point in time, Jennifer Stewart said that people trust that others care and are doing their job well.\n'],
   0.9978841145833334)])

In [17]:
# Evaluation loop

experiment_id = 'cluster_claims_nli_r2'
output_file = f'./data/results.{experiment_id}.json'
resume = True

# Allow resuming from previous results
if resume and os.path.exists(output_file):
	results = json.load(open(output_file, "r"))
	
	# Runs store lists instead of dicts, convert them
	if type(results) is list:
		results = { x['unique_id']: x for x in results }
else:
	results = { }

def write_output():
	# Write output file
	with open(output_file, "w") as f:
		json.dump(list(results.values()), f, indent=2)


# Create flat list of examples, shuffle to improve live stats accuracy
test_set = list(all_examples.items())
random.shuffle(test_set)

# Enumerate to get indices, convert to list to get progress indication
test_set = list(enumerate(test_set))

for i, (unique_id, example) in tqdm.tqdm(test_set):
	unique_id = example["unique id"]
	text = example["text"]
	y_true = example["label"]        # 0 or 1
	doc_type = example["doc_type"]   # story/wiki/news

	# Skip tests already completed previous run
	if unique_id in results:
		continue

	p_pred, evidence_max, all_evidence = detect(unique_id, text, DETECT_MODE, False)

	# Some examples fail due to claim extraction returning 0 or 1 claims
	if p_pred is None:
		print(f'Warning! Detection failed for {unique_id}')
		continue

	results[unique_id] = {
		"unique_id": unique_id,
		"p_pred": p_pred,
		"y_true": y_true,
		"doc_type": doc_type
	}

	# Ocassionally write output, to get live stats and in case of crash
	if i % 10 == 0:
		write_output()

write_output()

 10%|█         | 93/891 [12:47<1:49:16,  8.22s/it]

Warning! Detection failed for wiki_train_29390


 33%|███▎      | 293/891 [38:29<59:56,  6.01s/it]  

Warning! Detection failed for wiki_train_29193


 37%|███▋      | 331/891 [43:03<1:25:25,  9.15s/it]

...

 99%|█████████▊| 878/891 [1:52:29<02:08,  9.92s/it]

Warning! Detection failed for 3503017442_8


 99%|█████████▉| 880/891 [1:52:45<01:39,  9.03s/it]

Warning! Detection failed for 3503017447_7


100%|██████████| 891/891 [1:54:15<00:00,  7.69s/it]
